## 04 — Geração determinística de positivos e negativos (v3)
Gera 3.000 positivos e 3.000 negativos em um diretório versionado pelo hash do
manifesto e da configuração. Cada amostra tem seed independente, permitindo retomada
idêntica. As OBBs são integralmente válidas no intervalo [0,1].
Fundos B são usados somente no treino e em proporção explícita. Os hard negatives desta
versão são sintéticos; não representam uma avaliação de artefatos reais.

In [1]:
%pip install -q opencv-python-headless pandas tqdm

from google.colab import drive
from pathlib import Path
import hashlib, json, math, os, shutil, time
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}", flush=True)

def copiar_arvore_com_progresso(origem, destino, descricao):
    origem, destino = Path(origem), Path(destino)
    arquivos = sorted(p for p in origem.rglob('*') if p.is_file())
    log_etapa(f'{descricao}: início ({len(arquivos)} arquivos)')
    inicio = time.perf_counter()
    for arquivo in tqdm(arquivos, desc=descricao, unit='arq'):
        alvo = destino / arquivo.relative_to(origem)
        alvo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(arquivo, alvo)
    log_etapa(f'{descricao}: concluído em {time.perf_counter()-inicio:.1f}s')

drive.mount('/content/drive', force_remount=True)
BACKUP_DIR = Path('/content/drive/MyDrive/tcc-satellite-streaks')
PROC = Path('data/processed_v3'); PROC.mkdir(parents=True, exist_ok=True)
BG = PROC/'backgrounds_curated'; BG.mkdir(parents=True, exist_ok=True)

for nome in ['manifest_split_fundos_v3.csv','contrato_split_v3.json']:
    shutil.copy2(BACKUP_DIR/'data/processed_v3'/nome, PROC/nome)
copiar_arvore_com_progresso(
    BACKUP_DIR/'data/processed_v3/backgrounds_curated', BG,
    'Restaurando fundos curados'
)
df_fundos = pd.read_csv(PROC/'manifest_split_fundos_v3.csv')

CONFIG = {
    'revisao_gerador': 3,
    'seed_mestra': 20260823,
    'total_positivos': 3000,
    'total_negativos': 3000,
    'split': {'train':0.70,'val':0.15,'test':0.15},
    'brilho': {'constante':0.60,'variavel':0.25,'tracejado':0.15},
    'negativos': {'limpo':0.80,'hard_sintetico':0.20},
    'fracao_B_no_treino': 0.20,
    'tamanho': 512,
    'n_segmentos': 96,
    'snr_pico': [3.0, 25.0],
    'sigma_ruido_augmentation': [0.0, 2.0],
}

bytes_contrato = (PROC/'manifest_split_fundos_v3.csv').read_bytes() + json.dumps(CONFIG,sort_keys=True).encode()
RUN_ID = 'synthetic_v3_' + hashlib.sha256(bytes_contrato).hexdigest()[:12]
RUN = Path('data/synthetic_runs')/RUN_ID
DRIVE_RUN = BACKUP_DIR/'data/synthetic_runs'/RUN_ID
if DRIVE_RUN.exists() and not RUN.exists():
    RUN.parent.mkdir(parents=True,exist_ok=True)
    copiar_arvore_com_progresso(DRIVE_RUN, RUN, 'Retomando RUN existente')
RUN.mkdir(parents=True, exist_ok=True)
print('RUN_ID:', RUN_ID)

Mounted at /content/drive
[23:14:38] Restaurando fundos curados: início (325 arquivos)


Restaurando fundos curados:   0%|          | 0/325 [00:00<?, ?arq/s]

[23:14:45] Restaurando fundos curados: concluído em 7.1s
RUN_ID: synthetic_v3_c41d5091de56


## 1. Verificar os fundos contra o manifesto congelado

In [2]:
def sha256_arquivo(caminho, bloco=1024*1024):
    h = hashlib.sha256()
    with open(caminho, 'rb') as arquivo:
        for parte in iter(lambda: arquivo.read(bloco), b''):
            h.update(parte)
    return h.hexdigest()

faltantes, divergentes = [], []
for linha in df_fundos.itertuples():
    caminho = BG/linha.arquivo_final
    if not caminho.exists(): faltantes.append(linha.arquivo_final)
    elif sha256_arquivo(caminho) != linha.sha256_processado: divergentes.append(linha.arquivo_final)
assert not faltantes and not divergentes, f'Fundos inválidos: faltantes={faltantes[:3]}, divergentes={divergentes[:3]}'
assert df_fundos.groupby('sha256_processado')['split'].nunique().max() == 1
assert not ((df_fundos.categoria=='B') & (df_fundos.split!='train')).any()
print('Integridade dos fundos confirmada.')

Integridade dos fundos confirmada.


## 2. Plano exato e balanceado

In [3]:
def seed_estavel(texto):
    material = f"{CONFIG['seed_mestra']}::{texto}".encode()
    return int(hashlib.sha256(material).hexdigest()[:16], 16) % (2**32)

def maior_resto(total, proporcoes):
    bruto = {k: total*v for k,v in proporcoes.items()}
    base = {k: int(np.floor(v)) for k,v in bruto.items()}
    faltam = total - sum(base.values())
    ordem = sorted(proporcoes, key=lambda k:(bruto[k]-base[k],k), reverse=True)
    for k in ordem[:faltam]: base[k] += 1
    return base

def montar_tipo(tipo, total, subtipos):
    linhas = []
    cont_split = maior_resto(total, CONFIG['split'])
    for split in ['train','val','test']:
        n = cont_split[split]
        cont_sub = maior_resto(n, subtipos)
        lista_sub = [sub for sub in subtipos for _ in range(cont_sub[sub])]
        rng = np.random.default_rng(seed_estavel(f'plano::{tipo}::{split}'))
        rng.shuffle(lista_sub)
        if split == 'train':
            n_b = int(round(n*CONFIG['fracao_B_no_treino']))
            categorias = ['B']*n_b + ['A']*(n-n_b)
            rng.shuffle(categorias)
        else:
            categorias = ['A']*n
        for local, (subtipo, categoria) in enumerate(zip(lista_sub, categorias)):
            id_amostra = f"{tipo[:3]}_{split}_{local:05d}"
            linhas.append({'id':id_amostra,'tipo':tipo,'subtipo':subtipo,
                           'split':split,'categoria_fundo_planejada':categoria,
                           'seed_amostra':seed_estavel(id_amostra)})
    return linhas

plano = pd.DataFrame(
    montar_tipo('positivo', CONFIG['total_positivos'], CONFIG['brilho']) +
    montar_tipo('negativo', CONFIG['total_negativos'], CONFIG['negativos'])
)

# Distribuição balanceada dos fundos dentro de cada split/categoria.
plano['fundo_origem'] = ''
for (split,categoria), indices in plano.groupby(['split','categoria_fundo_planejada']).groups.items():
    fundos = df_fundos[(df_fundos.split==split)&(df_fundos.categoria==categoria)].sort_values('sha256_processado')
    assert len(fundos) > 0, f'Sem fundos para {split}/{categoria}'
    nomes = fundos['arquivo_final'].tolist()
    rng = np.random.default_rng(seed_estavel(f'fundos::{split}::{categoria}'))
    atribuicao = []
    while len(atribuicao) < len(indices):
        ciclo = nomes.copy(); rng.shuffle(ciclo); atribuicao.extend(ciclo)
    plano.loc[list(indices),'fundo_origem'] = atribuicao[:len(indices)]

mapa_meta = df_fundos.set_index('arquivo_final').to_dict('index')
print(pd.crosstab([plano.split,plano.tipo],plano.subtipo))
print('\nCategoria de fundo por split:')
print(pd.crosstab(plano.split,plano.categoria_fundo_planejada))

subtipo         constante  hard_sintetico  limpo  tracejado  variavel
split tipo                                                           
test  negativo          0              90    360          0         0
      positivo        270               0      0         67       113
train negativo          0             420   1680          0         0
      positivo       1260               0      0        315       525
val   negativo          0              90    360          0         0
      positivo        270               0      0         67       113

Categoria de fundo por split:
categoria_fundo_planejada     A    B
split                               
test                        900    0
train                      3360  840
val                         900    0


In [4]:
print(RUN_ID)

synthetic_v3_c41d5091de56


## 3. Renderização e OBB integralmente interna
O brilho é definido por SNR de pico aproximada em relação ao MAD robusto do fundo. Ruído
adicional é tratado explicitamente como augmentation gaussiana, sem alegação de modelo
físico do detector.

In [5]:
def perfil_brilho(t, modo, amplitude, n_ciclos):
    if modo == 'constante': return 1.0
    onda = 0.5*(1+math.sin(2*math.pi*n_ciclos*t))
    if modo == 'variavel': return 1-amplitude+2*amplitude*onda
    if modo == 'tracejado': return onda if onda > 0.15 else 0.0
    raise ValueError(modo)

def sigma_robusto(img):
    med = np.median(img)
    return max(1.0, 1.4826*np.median(np.abs(img.astype(float)-med)))

def aumentar_fundo(img, rng):
    flip_h, flip_v = bool(rng.integers(0,2)), bool(rng.integers(0,2))
    if flip_h: img = cv2.flip(img,1)
    if flip_v: img = cv2.flip(img,0)
    ang = float(rng.uniform(-15,15))
    h,w = img.shape[:2]
    matriz = cv2.getRotationMatrix2D((w/2,h/2),ang,1.0)
    img = cv2.warpAffine(img,matriz,(w,h),borderMode=cv2.BORDER_REFLECT)
    return img, {'flip_h':flip_h,'flip_v':flip_v,'rotacao_fundo_deg':ang}

def limite_ate_borda(cx,cy,ux,uy,w,h,margem,sinal):
    dx,dy = sinal*ux,sinal*uy
    limites=[]
    if dx>0: limites.append((w-1-margem-cx)/dx)
    elif dx<0: limites.append((margem-cx)/dx)
    if dy>0: limites.append((h-1-margem-cy)/dy)
    elif dy<0: limites.append((margem-cy)/dy)
    return min(v for v in limites if v>=0)

def sortear_segmento_interno(w,h,rng,meia_largura):
    margem = float(math.ceil(meia_largura+2))
    diagonal = math.hypot(w,h)
    for _ in range(500):
        cx,cy = rng.uniform(margem,w-1-margem),rng.uniform(margem,h-1-margem)
        ang = rng.uniform(0,math.pi)
        ux,uy = math.cos(ang),math.sin(ang)
        t_pos = limite_ate_borda(cx,cy,ux,uy,w,h,margem,1)
        t_neg = limite_ate_borda(cx,cy,ux,uy,w,h,margem,-1)
        f_pos,f_neg = rng.uniform(0.55,0.98,2)
        p0=(cx-ux*t_neg*f_neg,cy-uy*t_neg*f_neg)
        p1=(cx+ux*t_pos*f_pos,cy+uy*t_pos*f_pos)
        if math.dist(p0,p1)>=0.25*diagonal:
            return p0,p1,math.degrees(ang)
    raise RuntimeError('Não foi possível sortear segmento interno.')

def cantos_obb(p0,p1,meia_largura,w,h):
    x0,y0=p0; x1,y1=p1; dx,dy=x1-x0,y1-y0
    comp=math.hypot(dx,dy); px,py=-dy/comp,dx/comp
    cantos=[(x0+px*meia_largura,y0+py*meia_largura),(x1+px*meia_largura,y1+py*meia_largura),
            (x1-px*meia_largura,y1-py*meia_largura),(x0-px*meia_largura,y0-py*meia_largura)]
    norm=[(x/w,y/h) for x,y in cantos]
    assert all(0<=v<=1 for ponto in norm for v in ponto)
    return norm

def render_trilha(img,rng,modo):
    h,w=img.shape[:2]
    esp=int(rng.integers(1,4)); blur=float(rng.uniform(0.8,2.2))
    meia=esp/2+2.5*blur
    p0,p1,ang=sortear_segmento_interno(w,h,rng,meia)
    amplitude=float(rng.uniform(0.3,0.5)); ciclos=float(rng.uniform(1.5,3.0))
    snr=float(rng.uniform(*CONFIG['snr_pico']))
    camada=np.zeros((h,w),np.float32)
    for i in range(CONFIG['n_segmentos']):
        ta,tb=i/CONFIG['n_segmentos'],(i+1)/CONFIG['n_segmentos']; tm=(ta+tb)/2
        intensidade=perfil_brilho(tm,modo,amplitude,ciclos)
        if intensidade<=0: continue
        a=(int(p0[0]+(p1[0]-p0[0])*ta),int(p0[1]+(p1[1]-p0[1])*ta))
        b=(int(p0[0]+(p1[0]-p0[0])*tb),int(p0[1]+(p1[1]-p0[1])*tb))
        cv2.line(camada,a,b,float(intensidade),esp,cv2.LINE_AA)
    camada=cv2.GaussianBlur(camada,(0,0),sigmaX=blur)
    pico_alvo=min(220.0,snr*sigma_robusto(img))
    camada*=pico_alvo/max(float(camada.max()),1e-6)
    resultado=np.clip(img.astype(np.float32)+camada,0,255)
    obb=cantos_obb(p0,p1,meia,w,h)
    return resultado,obb,{'angulo_trilha_deg':ang,'comprimento_px':math.dist(p0,p1),
        'espessura_px':esp,'desfoque_sigma':blur,'amplitude':amplitude,
        'n_ciclos':ciclos,'snr_pico_alvo':snr,'p0':list(p0),'p1':list(p1)}

def render_pico_sintetico(img,rng):
    h,w=img.shape[:2]; camada=np.zeros((h,w),np.float32)
    cx,cy=int(rng.uniform(.2,.8)*w),int(rng.uniform(.2,.8)*h)
    comprimento=float(rng.uniform(.12,.30)*math.hypot(w,h)); base=float(rng.uniform(0,90))
    esp=int(rng.integers(1,4)); blur=float(rng.uniform(1.2,2.2)); brilho=float(rng.uniform(40,180))
    for offset in [0,90,180,270]:
        a=math.radians(base+offset); ponta=(cx+math.cos(a)*comprimento,cy+math.sin(a)*comprimento)
        for i in range(24):
            t0,t1=i/24,(i+1)/24; intensidade=brilho*(1-(t0+t1)/2)**1.5
            p0=(int(cx+(ponta[0]-cx)*t0),int(cy+(ponta[1]-cy)*t0))
            p1=(int(cx+(ponta[0]-cx)*t1),int(cy+(ponta[1]-cy)*t1))
            cv2.line(camada,p0,p1,float(intensidade),esp,cv2.LINE_AA)
    cv2.circle(camada,(cx,cy),2*esp,255,-1)
    camada=cv2.GaussianBlur(camada,(0,0),sigmaX=blur)
    return np.clip(img.astype(np.float32)+camada,0,255),{
        'pico_centro':[cx,cy],'pico_comprimento':comprimento,'pico_angulo_base':base,
        'pico_espessura':esp,'pico_blur':blur,'pico_brilho':brilho}

def ruido_augmentation(img,rng):
    sigma=float(rng.uniform(*CONFIG['sigma_ruido_augmentation']))
    if sigma>0: img=img+rng.normal(0,sigma,img.shape)
    return np.clip(img,0,255).astype(np.uint8),sigma

## 4. Geração retomável por amostra
Se uma execução for interrompida, cada ID é regenerado de forma idêntica. O manifesto
parcial é salvo junto das imagens; não há dependência da posição atual do RNG global.

In [6]:
for split in ['train','val','test']:
    (RUN/'images'/split).mkdir(parents=True,exist_ok=True)
    (RUN/'labels'/split).mkdir(parents=True,exist_ok=True)

caminho_manifest=RUN/'manifest_geracao_v3.csv'
registros_existentes={}
if caminho_manifest.exists():
    antigo=pd.read_csv(caminho_manifest)
    registros_existentes={r['id']:r for r in antigo.to_dict('records')}
registros=[]
arquivos_pendentes_backup=[]

def salvar_label(caminho,obb):
    if obb is None: caminho.write_text('',encoding='utf-8'); return
    valores=' '.join(f'{v:.8f}' for ponto in obb for v in ponto)
    caminho.write_text(f'0 {valores}\n',encoding='utf-8')

def salvar_manifesto_atomico(registros):
    tmp=caminho_manifest.with_suffix('.tmp.csv')
    pd.DataFrame(registros).to_csv(tmp,index=False)
    tmp.replace(caminho_manifest)

def sincronizar_pendentes(descricao):
    unicos = sorted(set(arquivos_pendentes_backup), key=lambda p: p.as_posix())
    if unicos:
        log_etapa(f'{descricao}: {len(unicos)} arquivos novos')
        for arquivo in tqdm(unicos, desc=descricao, unit='arq'):
            destino = DRIVE_RUN / arquivo.relative_to(RUN)
            destino.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(arquivo, destino)
        arquivos_pendentes_backup.clear()
    if caminho_manifest.exists():
        DRIVE_RUN.mkdir(parents=True, exist_ok=True)
        shutil.copy2(caminho_manifest, DRIVE_RUN/caminho_manifest.name)

for numero,linha in enumerate(tqdm(plano.to_dict('records'),desc='Gerando v3')):
    img_path=RUN/'images'/linha['split']/(linha['id']+'.png')
    lbl_path=RUN/'labels'/linha['split']/(linha['id']+'.txt')
    anterior=registros_existentes.get(linha['id'])
    if anterior and img_path.exists() and lbl_path.exists() and anterior.get('fundo_origem')==linha['fundo_origem']:
        if sha256_arquivo(img_path)==anterior.get('sha256_imagem') and sha256_arquivo(lbl_path)==anterior.get('sha256_label'):
            registros.append(anterior); continue

    rng=np.random.default_rng(int(linha['seed_amostra']))
    fundo=cv2.imread(str(BG/linha['fundo_origem']),cv2.IMREAD_GRAYSCALE)
    assert fundo is not None
    fundo,meta_aug=aumentar_fundo(fundo,rng)
    params={}
    if linha['tipo']=='positivo':
        resultado,obb,params=render_trilha(fundo,rng,linha['subtipo'])
    elif linha['subtipo']=='hard_sintetico':
        resultado,params=render_pico_sintetico(fundo,rng); obb=None
    else:
        resultado=fundo.astype(np.float32); obb=None
    resultado,sigma_aug=ruido_augmentation(resultado,rng)
    cv2.imwrite(str(img_path),resultado); salvar_label(lbl_path,obb)
    meta_fundo=mapa_meta[linha['fundo_origem']]
    registro={**linha,'id_fundo':meta_fundo['id_fundo'],'fonte_fundo':meta_fundo['fonte'],
        'categoria_fundo':meta_fundo['categoria'],'sha256_fundo':meta_fundo['sha256_processado'],
        **meta_aug,**params,'sigma_ruido_augmentation':sigma_aug,
        'sha256_imagem':sha256_arquivo(img_path),'sha256_label':sha256_arquivo(lbl_path)}
    registros.append(registro)
    arquivos_pendentes_backup.extend([img_path, lbl_path])
    if (numero+1)%200==0:
        salvar_manifesto_atomico(registros)
        sincronizar_pendentes(f'Checkpoint {numero+1}/{len(plano)}')

salvar_manifesto_atomico(registros)
(RUN/'config_geracao_v3.json').write_text(json.dumps(CONFIG,indent=2),encoding='utf-8')
(RUN/'run_id.txt').write_text(RUN_ID+'\n',encoding='utf-8')
arquivos_pendentes_backup.extend([RUN/'config_geracao_v3.json', RUN/'run_id.txt'])

# Remove sobras apenas dentro deste RUN_ID versionado.
esperados={(r['split'],r['id']) for r in registros}
for split in ['train','val','test']:
    for p in (RUN/'images'/split).glob('*.png'):
        if (split,p.stem) not in esperados: p.unlink()
    for p in (RUN/'labels'/split).glob('*.txt'):
        if (split,p.stem) not in esperados: p.unlink()

sincronizar_pendentes('Backup final dos arquivos novos')
# Remove sobras remotas somente dentro deste RUN_ID versionado.
for split in ['train','val','test']:
    for pasta,sufixo in [('images','.png'),('labels','.txt')]:
        remoto=DRIVE_RUN/pasta/split
        if remoto.exists():
            for p in remoto.glob('*'+sufixo):
                if (split,p.stem) not in esperados: p.unlink()
print('Geração concluída localmente:',RUN_ID,len(registros))
print('O ponteiro LATEST_RUN_V3 ainda NÃO foi publicado; execute o gate rápido.')

Gerando v3:   0%|          | 0/6000 [00:00<?, ?it/s]

[23:17:39] Checkpoint 200/6000: 400 arquivos novos


Checkpoint 200/6000:   0%|          | 0/400 [00:00<?, ?arq/s]

[23:17:53] Checkpoint 400/6000: 400 arquivos novos


Checkpoint 400/6000:   0%|          | 0/400 [00:00<?, ?arq/s]

...

[23:24:44] Checkpoint 6000/6000: 400 arquivos novos


Checkpoint 6000/6000:   0%|          | 0/400 [00:00<?, ?arq/s]

[23:24:53] Backup final dos arquivos novos: 2 arquivos novos


Backup final dos arquivos novos:   0%|          | 0/2 [00:00<?, ?arq/s]

Geração concluída localmente: synthetic_v3_c41d5091de56 6000
O ponteiro LATEST_RUN_V3 ainda NÃO foi publicado; execute o gate rápido.


## 5. Gate rápido

In [7]:
df_final=pd.read_csv(RUN/'manifest_geracao_v3.csv')
assert len(df_final)==CONFIG['total_positivos']+CONFIG['total_negativos']
assert df_final['id'].is_unique
assert (df_final.tipo=='positivo').sum()==CONFIG['total_positivos']
assert (df_final.tipo=='negativo').sum()==CONFIG['total_negativos']
assert not ((df_final.categoria_fundo=='B')&(df_final.split!='train')).any()
assert df_final.groupby('sha256_fundo')['split'].nunique().max()==1
print(pd.crosstab([df_final.split,df_final.tipo],df_final.subtipo))
print('\nUso de fundos únicos:',df_final.groupby('split')['sha256_fundo'].nunique().to_dict())

# Somente um RUN aprovado pelo gate se torna a versão corrente para os notebooks seguintes.
ponteiro_local=Path('data/synthetic_runs/LATEST_RUN_V3.txt')
ponteiro_local.parent.mkdir(parents=True,exist_ok=True)
ponteiro_local.write_text(RUN_ID+'\n',encoding='utf-8')
ponteiro_drive=BACKUP_DIR/'data/synthetic_runs/LATEST_RUN_V3.txt'
ponteiro_drive.parent.mkdir(parents=True,exist_ok=True)
shutil.copy2(ponteiro_local,ponteiro_drive)
log_etapa(f'Gate rápido aprovado; ponteiro publicado para {RUN_ID}')

subtipo         constante  hard_sintetico  limpo  tracejado  variavel
split tipo                                                           
test  negativo          0              90    360          0         0
      positivo        270               0      0         67       113
train negativo          0             420   1680          0         0
      positivo       1260               0      0        315       525
val   negativo          0              90    360          0         0
      positivo        270               0      0         67       113

Uso de fundos únicos: {'test': 40, 'train': 245, 'val': 40}
[23:33:09] Gate rápido aprovado; ponteiro publicado para synthetic_v3_c41d5091de56
